In [ ]:
# -*- coding: utf-8 -*-
"""
FINAL MODEL COMPARISON & ROBUSTNESS SCRIPT
Amaç:
- Nihai aday modeli (Extra Trees) aynı feature set ile diğer güçlü adaylarla karşılaştırmak
- Çoklu random_state ile sağlamlık testi yapmak
- Final seed=42 için detaylı tahmin tablosu, importance ve özet CSV üretmek

Değerlendirme rejimi:
- RANDOM SPLIT (user-accepted regime)
- Bu script strict time-series forecast iddiası için değil,
  random-split genel tahmin başarısı için kullanılmalıdır.
"""

import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ============================================================
# 0) DOSYA BUL
# ============================================================
def locate_excel():
    preferred = [
        "/content/Veri_Seti_Son_duzeltilmis_v3_agac_farklari_eklendi.xlsx",
        "/content/Veri_Seti_Son_duzeltilmis_v3_yeniden_olusturuldu.xlsx",
        "/content/Veri Seti Son.xlsx",
        "/mnt/data/Veri_Seti_Son_duzeltilmis_v3_agac_farklari_eklendi.xlsx",
        "/mnt/data/Veri_Seti_Son_duzeltilmis_v3_yeniden_olusturuldu.xlsx",
        "/mnt/data/Veri Seti Son.xlsx",
    ]
    for p in preferred:
        if os.path.exists(p):
            return p

    candidates = (
        glob.glob("/content/*.xlsx") + glob.glob("/content/*.xls")
        + glob.glob("/mnt/data/*.xlsx") + glob.glob("/mnt/data/*.xls")
        + glob.glob("./*.xlsx") + glob.glob("./*.xls")
    )
    if not candidates:
        raise FileNotFoundError("Excel dosyası bulunamadı.")
    return candidates[0]

FILE_PATH = locate_excel()
print("Kullanılan dosya:", FILE_PATH)

# ============================================================
# 1) VERİYİ OKU
# ============================================================
try:
    df = pd.read_excel(FILE_PATH, sheet_name="Tam veri seti")
except Exception:
    df = pd.read_excel(FILE_PATH)

df.columns = [c.strip() for c in df.columns]

if "YIL" in df.columns:
    df = df.sort_values("YIL", ascending=True).reset_index(drop=True)

# ============================================================
# 2) GECİKME VE FEATURE SET
# ============================================================
if "TOPLAM MEYVE VEREN AĞAÇ SAYISI" not in df.columns:
    raise ValueError("Beklenen kolon bulunamadı: TOPLAM MEYVE VEREN AĞAÇ SAYISI")

df["AGAC_SAYISI_L1"] = df["TOPLAM MEYVE VEREN AĞAÇ SAYISI"].shift(1)
df = df.dropna(subset=["AGAC_SAYISI_L1"]).reset_index(drop=True)

TARGET = "TOPLAM ÜRETİM(YAŞ KAYISI/TON)"

FEATURES = [
    "TOPLAM MEYVE VEREN AĞAÇ SAYISI",
    "AGAC_SAYISI_L1",
    "Haziran ayı ortalama sıcaklık",
    "Temmuz ayı sıcaklık ortalaması",
    "Haziran max sıcaklık",
    "Temmuz ayı max sıcaklık",
    "Haziran ayı rüzgar hızı ortalaması",
    "Temmuz ayı rüzgar hızı ortalaması",
]

missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Eksik feature(lar): {missing}")

for forbidden in ["AGAC_BASI_VERIM_KG_AGAC", "FELAKET_YILI_UZMAN", "FELAKET_YILI_KURAL", "FELAKET_NOTU", TARGET, "YIL"]:
    if forbidden in FEATURES:
        raise ValueError(f"Forbidden feature detected: {forbidden}")

X = df[FEATURES].copy()
y = df[TARGET].copy()

# ============================================================
# 3) METRİK FONKSİYONU
# ============================================================
def metric_dict(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mse)),
        "MSE": float(mse),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
    }

def print_metrics(title, y_true, y_pred):
    m = metric_dict(y_true, y_pred)
    print(f"\n{title}")
    print(f"R²   : {m['R2']:.4f}")
    print(f"RMSE : {m['RMSE']:.2f}")
    print(f"MSE  : {m['MSE']:.2f}")
    print(f"MAE  : {m['MAE']:.2f}")

# ============================================================
# 4) MODELLER
# ============================================================
def build_models(seed):
    models = {}

    models["ExtraTrees"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=500,
            max_depth=6,
            random_state=seed
        ))
    ])

    models["RandomForest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=5,
            random_state=seed
        ))
    ])

    models["SVR_best"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR(
            kernel="linear",
            C=5000,
            epsilon=0.5,
            gamma="scale"
        ))
    ])

    return models

# ============================================================
# 5) ÇOKLU SEED SAĞLAMLIK TESTİ
# ============================================================
SEEDS = [10, 21, 42, 77, 99]
rows = []

for seed in SEEDS:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=seed, shuffle=True
    )

    models = build_models(seed)

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        m_tr = metric_dict(y_train, pred_train)
        m_te = metric_dict(y_test, pred_test)

        rows.append({
            "seed": seed,
            "model": model_name,
            "train_R2": m_tr["R2"],
            "train_RMSE": m_tr["RMSE"],
            "train_MSE": m_tr["MSE"],
            "train_MAE": m_tr["MAE"],
            "test_R2": m_te["R2"],
            "test_RMSE": m_te["RMSE"],
            "test_MSE": m_te["MSE"],
            "test_MAE": m_te["MAE"],
        })

robust_df = pd.DataFrame(rows)

summary_df = robust_df.groupby("model").agg(
    train_R2_mean=("train_R2", "mean"),
    train_R2_std=("train_R2", "std"),
    test_R2_mean=("test_R2", "mean"),
    test_R2_std=("test_R2", "std"),
    test_RMSE_mean=("test_RMSE", "mean"),
    test_RMSE_std=("test_RMSE", "std"),
    test_MAE_mean=("test_MAE", "mean"),
    test_MAE_std=("test_MAE", "std"),
).reset_index()

summary_df = summary_df.sort_values(["test_R2_mean", "train_R2_mean"], ascending=False).reset_index(drop=True)

print("\nÇOKLU SEED SAĞLAMLIK ÖZETİ")
print(summary_df)

# ============================================================
# 6) FINAL SEED = 42 DETAY RAPORU
# ============================================================
FINAL_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=FINAL_SEED, shuffle=True
)

models = build_models(FINAL_SEED)

final_metric_rows = []
prediction_frames = []
importance_frames = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    print_metrics(f"{model_name} - EĞİTİM", y_train, pred_train)
    print_metrics(f"{model_name} - TEST", y_test, pred_test)

    m_tr = metric_dict(y_train, pred_train)
    m_te = metric_dict(y_test, pred_test)

    final_metric_rows.append({"model": model_name, "dataset": "train", **m_tr})
    final_metric_rows.append({"model": model_name, "dataset": "test", **m_te})

    frame = pd.DataFrame({
        "Gercek_Ton": y_test.values,
        f"Tahmin_{model_name}": pred_test
    }, index=y_test.index)
    prediction_frames.append(frame)

    # Feature importance (yalnız ağaç modelleri)
    try:
        feature_importance = model.named_steps["model"].feature_importances_
        imp_df = pd.DataFrame({
            "model": model_name,
            "feature": FEATURES,
            "importance": feature_importance
        }).sort_values("importance", ascending=False)
        importance_frames.append(imp_df)
    except Exception:
        pass

final_metrics_df = pd.DataFrame(final_metric_rows)

pred_table = pd.concat(prediction_frames, axis=1)
pred_table = pred_table.loc[:, ~pred_table.columns.duplicated()].sort_index()

if "YIL" in df.columns:
    pred_table["YIL"] = df.loc[pred_table.index, "YIL"].values
    ordered_cols = ["YIL", "Gercek_Ton"] + [c for c in pred_table.columns if c not in ["YIL", "Gercek_Ton"]]
    pred_table = pred_table[ordered_cols]

# Extra Trees odaklı hata kolonları
if "Tahmin_ExtraTrees" in pred_table.columns:
    pred_table["Fark_Ton_ExtraTrees"] = pred_table["Gercek_Ton"] - pred_table["Tahmin_ExtraTrees"]
    pred_table["Hata_%_ExtraTrees"] = 100.0 * pred_table["Fark_Ton_ExtraTrees"] / np.maximum(pred_table["Gercek_Ton"], 1e-9)

print("\nFINAL SEED=42 TEST TAHMİN TABLOSU")
print(pred_table)

feature_importance_df = pd.concat(importance_frames, axis=0) if importance_frames else pd.DataFrame()

# ============================================================
# 7) GRAFİK
# ============================================================
plt.figure(figsize=(10, 6))
if "Tahmin_ExtraTrees" in pred_table.columns:
    plt.scatter(pred_table["Gercek_Ton"], pred_table["Tahmin_ExtraTrees"], label="ExtraTrees")
if "Tahmin_RandomForest" in pred_table.columns:
    plt.scatter(pred_table["Gercek_Ton"], pred_table["Tahmin_RandomForest"], label="RandomForest")
if "Tahmin_SVR_best" in pred_table.columns:
    plt.scatter(pred_table["Gercek_Ton"], pred_table["Tahmin_SVR_best"], label="SVR_best")

min_val = float(pred_table["Gercek_Ton"].min())
max_val = float(pred_table["Gercek_Ton"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Gerçek ton")
plt.ylabel("Tahmin ton")
plt.title("Gerçek vs Tahmin")
plt.legend()
plt.tight_layout()

plot_path = "/content/final_model_comparison_plot.png"
plt.savefig(plot_path, dpi=150)
plt.close()

# ============================================================
# 8) KAYDET
# ============================================================
robust_path = "/content/final_model_robustness_by_seed.csv"
summary_path = "/content/final_model_robustness_summary.csv"
metrics_path = "/content/final_model_metrics_seed42.csv"
pred_path = "/content/final_model_predictions_seed42.csv"
imp_path = "/content/final_model_feature_importance.csv"

robust_df.to_csv(robust_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
final_metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
pred_table.to_csv(pred_path, index=False, encoding="utf-8-sig")
if not feature_importance_df.empty:
    feature_importance_df.to_csv(imp_path, index=False, encoding="utf-8-sig")

print("\nKaydedildi:")
print(robust_path)
print(summary_path)
print(metrics_path)
print(pred_path)
print(plot_path)
if not feature_importance_df.empty:
    print(imp_path)

# ============================================================
# 9) DEĞİŞKEN GRUPLARINA GÖRE KARŞILAŞTIRMALI ANALİZ
# ============================================================

FEATURE_GROUPS = {
    "Kapasite": [
        "TOPLAM MEYVE VEREN AĞAÇ SAYISI",
        "AGAC_SAYISI_L1",
    ],
    "Meteoroloji": [
        "Haziran ayı ortalama sıcaklık",
        "Temmuz ayı sıcaklık ortalaması",
        "Haziran max sıcaklık",
        "Temmuz ayı max sıcaklık",
        "Haziran ayı rüzgar hızı ortalaması",
        "Temmuz ayı rüzgar hızı ortalaması",
    ],
    "Birlesik_8": FEATURES.copy(),
}

# Grup sütunlarının mevcut X matrisiyle birebir uyumunu kontrol et.
for group_name, group_features in FEATURE_GROUPS.items():
    missing_group_features = [c for c in group_features if c not in X.columns]
    if missing_group_features:
        raise ValueError(
            f"{group_name} grubunda eksik değişken(ler): {missing_group_features}"
        )

feature_group_rows = []

for seed in SEEDS:
    # Aynı seed altında tüm değişken grupları ve tüm modeller
    # birebir aynı eğitim/test gözlemlerini kullanır.
    row_indices = np.arange(len(X))
    train_idx, test_idx = train_test_split(
        row_indices,
        test_size=0.20,
        random_state=seed,
        shuffle=True
    )

    y_train_group = y.iloc[train_idx]
    y_test_group = y.iloc[test_idx]

    for group_name, group_features in FEATURE_GROUPS.items():
        X_group = X[group_features].copy()
        X_train_group = X_group.iloc[train_idx]
        X_test_group = X_group.iloc[test_idx]

        group_models = build_models(seed)

        for model_name, model in group_models.items():
            model.fit(X_train_group, y_train_group)

            pred_train_group = model.predict(X_train_group)
            pred_test_group = model.predict(X_test_group)

            m_train_group = metric_dict(y_train_group, pred_train_group)
            m_test_group = metric_dict(y_test_group, pred_test_group)

            feature_group_rows.append({
                "seed": seed,
                "feature_group": group_name,
                "feature_count": len(group_features),
                "model": model_name,
                "train_R2": m_train_group["R2"],
                "train_RMSE": m_train_group["RMSE"],
                "train_MSE": m_train_group["MSE"],
                "train_MAE": m_train_group["MAE"],
                "test_R2": m_test_group["R2"],
                "test_RMSE": m_test_group["RMSE"],
                "test_MSE": m_test_group["MSE"],
                "test_MAE": m_test_group["MAE"],
            })

# ============================================================
# 10) DEĞİŞKEN GRUBU - DETAYLI SONUÇ TABLOSU
# ============================================================

feature_group_results_df = pd.DataFrame(feature_group_rows)

print("\n" + "=" * 100)
print("DEĞİŞKEN GRUPLARINA GÖRE TÜM SEED SONUÇLARI")
print("=" * 100)
print(
    feature_group_results_df[
        [
            "seed",
            "feature_group",
            "feature_count",
            "model",
            "test_R2",
            "test_RMSE",
            "test_MAE",
        ]
    ].to_string(index=False)
)

# ============================================================
# 11) DEĞİŞKEN GRUPLARI - 5 SEED ÖZETİ
# ============================================================

feature_group_summary_df = (
    feature_group_results_df
    .groupby(["feature_group", "feature_count", "model"])
    .agg(
        train_R2_mean=("train_R2", "mean"),
        train_R2_std=("train_R2", "std"),
        test_R2_mean=("test_R2", "mean"),
        test_R2_std=("test_R2", "std"),
        test_RMSE_mean=("test_RMSE", "mean"),
        test_RMSE_std=("test_RMSE", "std"),
        test_MAE_mean=("test_MAE", "mean"),
        test_MAE_std=("test_MAE", "std"),
    )
    .reset_index()
)

# Sadece okunabilir çıktı için sıralama; metodolojik seçim ölçütü değildir.
feature_group_summary_df = feature_group_summary_df.sort_values(
    ["model", "feature_group"]
).reset_index(drop=True)

print("\n" + "=" * 100)
print("DEĞİŞKEN GRUPLARI - 5 SEED ORTALAMA / STANDART SAPMA")
print("=" * 100)
print(
    feature_group_summary_df[
        [
            "feature_group",
            "feature_count",
            "model",
            "test_R2_mean",
            "test_R2_std",
            "test_RMSE_mean",
            "test_RMSE_std",
            "test_MAE_mean",
            "test_MAE_std",
        ]
    ].to_string(index=False)
)

# ============================================================
# 12) FINAL SEED = 42 - DEĞİŞKEN GRUBU KARŞILAŞTIRMASI
# ============================================================

feature_group_seed42_df = (
    feature_group_results_df[
        feature_group_results_df["seed"] == FINAL_SEED
    ]
    .copy()
    .sort_values(["model", "feature_group"])
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("FINAL SEED = 42 - DEĞİŞKEN GRUBU KARŞILAŞTIRMASI")
print("=" * 100)
print(
    feature_group_seed42_df[
        [
            "feature_group",
            "feature_count",
            "model",
            "train_R2",
            "test_R2",
            "test_RMSE",
            "test_MAE",
        ]
    ].to_string(index=False)
)

# ============================================================
# 13) DEĞİŞKEN GRUBU SONUÇLARINI KAYDET
# ============================================================

group_results_path = "/content/feature_group_results_by_seed.csv"
group_summary_path = "/content/feature_group_summary.csv"
group_seed42_path = "/content/feature_group_seed42.csv"

feature_group_results_df.to_csv(
    group_results_path,
    index=False,
    encoding="utf-8-sig"
)

feature_group_summary_df.to_csv(
    group_summary_path,
    index=False,
    encoding="utf-8-sig"
)

feature_group_seed42_df.to_csv(
    group_seed42_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDeğişken grubu analiz dosyaları kaydedildi:")
print(group_results_path)
print(group_summary_path)
print(group_seed42_path)
# ============================================================
# 14) GECİKMELİ AĞAÇ SAYISI (L1) DUYARLILIK ANALİZİ
# ============================================================

LAG_FEATURE_GROUPS = {
    "L1_Yok": [
        "TOPLAM MEYVE VEREN AĞAÇ SAYISI",
        "Haziran ayı ortalama sıcaklık",
        "Temmuz ayı sıcaklık ortalaması",
        "Haziran max sıcaklık",
        "Temmuz ayı max sıcaklık",
        "Haziran ayı rüzgar hızı ortalaması",
        "Temmuz ayı rüzgar hızı ortalaması",
    ],

    "L1_Var": [
        "TOPLAM MEYVE VEREN AĞAÇ SAYISI",
        "AGAC_SAYISI_L1",
        "Haziran ayı ortalama sıcaklık",
        "Temmuz ayı sıcaklık ortalaması",
        "Haziran max sıcaklık",
        "Temmuz ayı max sıcaklık",
        "Haziran ayı rüzgar hızı ortalaması",
        "Temmuz ayı rüzgar hızı ortalaması",
    ],
}

# Sütun kontrolü
for lag_group_name, lag_features in LAG_FEATURE_GROUPS.items():
    missing_lag_features = [c for c in lag_features if c not in X.columns]

    if missing_lag_features:
        raise ValueError(
            f"{lag_group_name} grubunda eksik değişken(ler): "
            f"{missing_lag_features}"
        )


lag_sensitivity_rows = []

for seed in SEEDS:

    # Her iki L1 koşulu birebir aynı eğitim/test gözlemlerini kullanır.
    row_indices = np.arange(len(X))

    train_idx, test_idx = train_test_split(
        row_indices,
        test_size=0.20,
        random_state=seed,
        shuffle=True
    )

    y_train_lag = y.iloc[train_idx]
    y_test_lag = y.iloc[test_idx]

    for lag_group_name, lag_features in LAG_FEATURE_GROUPS.items():

        X_lag = X[lag_features].copy()

        X_train_lag = X_lag.iloc[train_idx]
        X_test_lag = X_lag.iloc[test_idx]

        lag_models = build_models(seed)

        for model_name, model in lag_models.items():

            model.fit(X_train_lag, y_train_lag)

            pred_train_lag = model.predict(X_train_lag)
            pred_test_lag = model.predict(X_test_lag)

            m_train_lag = metric_dict(y_train_lag, pred_train_lag)
            m_test_lag = metric_dict(y_test_lag, pred_test_lag)

            lag_sensitivity_rows.append({
                "seed": seed,
                "lag_condition": lag_group_name,
                "feature_count": len(lag_features),
                "model": model_name,

                "train_R2": m_train_lag["R2"],
                "train_RMSE": m_train_lag["RMSE"],
                "train_MSE": m_train_lag["MSE"],
                "train_MAE": m_train_lag["MAE"],

                "test_R2": m_test_lag["R2"],
                "test_RMSE": m_test_lag["RMSE"],
                "test_MSE": m_test_lag["MSE"],
                "test_MAE": m_test_lag["MAE"],
            })


# ============================================================
# 15) L1 DUYARLILIK - TÜM SEED SONUÇLARI
# ============================================================

lag_sensitivity_results_df = pd.DataFrame(lag_sensitivity_rows)

print("\n" + "=" * 100)
print("GECİKMELİ AĞAÇ SAYISI (L1) - TÜM SEED SONUÇLARI")
print("=" * 100)

print(
    lag_sensitivity_results_df[
        [
            "seed",
            "lag_condition",
            "feature_count",
            "model",
            "test_R2",
            "test_RMSE",
            "test_MAE",
        ]
    ].to_string(index=False)
)


# ============================================================
# 16) L1 DUYARLILIK - 5 SEED ÖZETİ
# ============================================================

lag_sensitivity_summary_df = (
    lag_sensitivity_results_df
    .groupby(["lag_condition", "feature_count", "model"])
    .agg(
        train_R2_mean=("train_R2", "mean"),
        train_R2_std=("train_R2", "std"),

        test_R2_mean=("test_R2", "mean"),
        test_R2_std=("test_R2", "std"),

        test_RMSE_mean=("test_RMSE", "mean"),
        test_RMSE_std=("test_RMSE", "std"),

        test_MAE_mean=("test_MAE", "mean"),
        test_MAE_std=("test_MAE", "std"),
    )
    .reset_index()
    .sort_values(["model", "lag_condition"])
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("L1 DUYARLILIK - 5 SEED ORTALAMA / STANDART SAPMA")
print("=" * 100)

print(
    lag_sensitivity_summary_df[
        [
            "lag_condition",
            "feature_count",
            "model",
            "test_R2_mean",
            "test_R2_std",
            "test_RMSE_mean",
            "test_RMSE_std",
            "test_MAE_mean",
            "test_MAE_std",
        ]
    ].to_string(index=False)
)


# ============================================================
# 17) FINAL SEED = 42 - L1 DUYARLILIK KARŞILAŞTIRMASI
# ============================================================

lag_sensitivity_seed42_df = (
    lag_sensitivity_results_df[
        lag_sensitivity_results_df["seed"] == FINAL_SEED
    ]
    .copy()
    .sort_values(["model", "lag_condition"])
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("FINAL SEED = 42 - L1 VAR / L1 YOK KARŞILAŞTIRMASI")
print("=" * 100)

print(
    lag_sensitivity_seed42_df[
        [
            "lag_condition",
            "feature_count",
            "model",
            "train_R2",
            "test_R2",
            "test_RMSE",
            "test_MAE",
        ]
    ].to_string(index=False)
)


# ============================================================
# 18) L1 DUYARLILIK SONUÇLARINI KAYDET
# ============================================================

lag_results_path = "/content/lag_sensitivity_results_by_seed.csv"
lag_summary_path = "/content/lag_sensitivity_summary.csv"
lag_seed42_path = "/content/lag_sensitivity_seed42.csv"

lag_sensitivity_results_df.to_csv(
    lag_results_path,
    index=False,
    encoding="utf-8-sig"
)

lag_sensitivity_summary_df.to_csv(
    lag_summary_path,
    index=False,
    encoding="utf-8-sig"
)

lag_sensitivity_seed42_df.to_csv(
    lag_seed42_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nL1 duyarlılık analiz dosyaları kaydedildi:")
print(lag_results_path)
print(lag_summary_path)
print(lag_seed42_path)

